# EV terrain generator for investigating Solomon style benchmarks

In [ ]:
import os
import json
import shutil
import numpy as np
import pandas as pd
import rasterio
from rasterio.merge import merge
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors

# ============================================================
# CONFIGURATION
# ============================================================

BASE_DIR      = "C:/Users/user/ML/Optimisation/VRPTW"
INSTANCES_DIR = f"{BASE_DIR}/instances"   # raw Solomon CSVs  - read only
DEM_DIR       = f"{BASE_DIR}/DEMS"        # Copernicus TIFFs  - read only
SCALED_DIR    = f"{BASE_DIR}/scaled"      # all generated output

SCALE_MULTIPLIER = 50    # 1 Solomon unit * 50 = metres
SUBSTEP_INTERVAL = 100   # metres between elevation sample points

cities = pd.read_csv(f"{BASE_DIR}/revised_cities.csv")

benchmarks = [
    "c107", "c108", "c109",
    "r101", "r104", "r105", "r204", "r205", "r208",
    "rc101", "rc104", "rc107", "rc201", "rc204", "rc207",
]

scales = [5, 10]

# ============================================================
# TILE INDEX HELPERS
# ============================================================

def lat_tile_index(lat):
    """
    SW corner latitude index for a 1°x1° Copernicus tile.
    UK latitudes are all positive so floor works correctly.
    """
    return int(np.floor(lat))

def lon_tile_index(lon):
    """
    SW corner longitude index for a 1°x1° Copernicus tile.
    For negative longitudes (W), floor gives the correct SW corner:
      lon=-1.89 → floor=-2 → W002 tile covers -2.0 to -1.0  ✓
      lon=-1.00 → floor=-1 → W001 tile covers -1.0 to  0.0  ✓
      lon= 0.12 → floor= 0 → E000 tile covers  0.0 to  1.0  ✓
    """
    return int(np.floor(lon))

def tile_name_from_index(lat_idx, lon_idx):
    """Build Copernicus tile name from SW corner indices."""
    lat_prefix = 'N' if lat_idx >= 0 else 'S'
    lon_prefix = 'E' if lon_idx >= 0 else 'W'
    return (f"Copernicus_DSM_COG_10_"
            f"{lat_prefix}{abs(lat_idx):02d}_00_"
            f"{lon_prefix}{abs(lon_idx):03d}_00_DEM")

# ============================================================
# DEM ACQUISITION
# ============================================================

def download_copernicus_tile(lat_idx, lon_idx, output_dir):
    """
    Download a single Copernicus GLO-30 tile by its SW corner indices.
    Returns local path or None if tile doesn't exist in dataset.
    """
    tile_name   = tile_name_from_index(lat_idx, lon_idx)
    s3_key      = f"{tile_name}/{tile_name}.tif"
    output_path = os.path.join(output_dir, f"{tile_name}.tif")

    if os.path.exists(output_path):
        print(f"    Already exists: {tile_name}")
        return output_path

    s3     = boto3.client('s3', config=Config(signature_version=UNSIGNED))
    bucket = 'copernicus-dem-30m'

    # Use list_objects_v2 to check existence — avoids ClientError handling issues
    response = s3.list_objects_v2(Bucket=bucket, Prefix=s3_key, MaxKeys=1)
    if 'Contents' not in response:
        print(f"    Not found in dataset, skipping: {tile_name}")
        return None

    print(f"    Downloading {tile_name}...")
    s3.download_file(bucket, s3_key, output_path)
    print(f"    Saved: {output_path}")
    return output_path


def get_tiles_for_city(city_name, lat, lon, output_dir, buffer_deg=0.15):
    """
    Determine which 1°x1° tiles cover the city bounding box
    and download any that aren't already present.
    """
    lat_min = lat - buffer_deg
    lat_max = lat + buffer_deg
    lon_min = lon - buffer_deg
    lon_max = lon + buffer_deg

    lat_indices = set(range(lat_tile_index(lat_min),
                            lat_tile_index(lat_max) + 1))
    lon_indices = set(range(lon_tile_index(lon_min),
                            lon_tile_index(lon_max) + 1))

    print(f"  Tiles needed for {city_name}: "
          f"lat={sorted(lat_indices)} lon={sorted(lon_indices)}")

    paths = []
    for la in lat_indices:
        for lo in lon_indices:
            path = download_copernicus_tile(la, lo, output_dir)
            if path is not None:
                paths.append(path)

    if not paths:
        raise RuntimeError(
            f"No DEM tiles found for {city_name} at lat={lat}, lon={lon}")
    return paths


def merge_tiles(tile_paths, output_path):
    """Merge multiple tiles into one, or copy if only one tile."""
    if os.path.exists(output_path):
        print(f"    Merged DEM already exists: {output_path}")
        return output_path

    if len(tile_paths) == 1:
        shutil.copy2(tile_paths[0], output_path)
        print(f"    Single tile copied to: {output_path}")
        return output_path

    datasets        = [rasterio.open(p) for p in tile_paths]
    mosaic, transform = merge(datasets)
    profile         = datasets[0].profile
    profile.update({
        "height":    mosaic.shape[1],
        "width":     mosaic.shape[2],
        "transform": transform
    })
    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(mosaic)
    for ds in datasets:
        ds.close()
    print(f"    Merged {len(tile_paths)} tiles → {output_path}")
    return output_path


def ensure_dem(city_name, lat, lon):
    city_dem_dir = os.path.join(DEM_DIR, city_name)
    os.makedirs(city_dem_dir, exist_ok=True)
    dem_path = os.path.join(city_dem_dir, f"{city_name}_dem.tif")

    if os.path.exists(dem_path):
        # Validate bounds in a separate block so file is fully closed before any delete
        needs_redownload = False
        with rasterio.open(dem_path) as src:
            b = src.bounds
            if not (b.left <= lon <= b.right and b.bottom <= lat <= b.top):
                print(f"  WARNING: existing DEM for {city_name} does not "
                      f"cover ({lat}, {lon}) — bounds are {b}")
                print(f"  Deleting and re-downloading...")
                needs_redownload = True
            else:
                print(f"  DEM ready: {dem_path}")

        # File is now closed — safe to delete on Windows
        if needs_redownload:
            os.remove(dem_path)
        else:
            return dem_path

    print(f"  Fetching DEM for {city_name}...")
    tile_paths = get_tiles_for_city(city_name, lat, lon, city_dem_dir)
    return merge_tiles(tile_paths, dem_path)


# ============================================================
# GEOMETRY
# ============================================================

def euclidean_distance(p1, p2):
    return np.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)


def prim_mst(points):
    n        = len(points)
    visited  = [False] * n
    min_edge = [(None, float('inf'))] * n
    min_edge[0] = (None, 0.0)
    mst = []
    for _ in range(n):
        u = min((i for i in range(n) if not visited[i]),
                key=lambda i: min_edge[i][1])
        visited[u] = True
        if min_edge[u][0] is not None:
            mst.append((min_edge[u][0], u, min_edge[u][1]))
        for v in range(n):
            if not visited[v]:
                w = euclidean_distance(points[u], points[v])
                if w < min_edge[v][1]:
                    min_edge[v] = (u, w)
    return mst


def compute_route(mst, n, start, end):
    """
    DFS path on MST from start to end.
    Returns intermediate nodes only — excludes destination,
    consistent with C++ MST::getSteps convention.
    """
    graph = {i: [] for i in range(n)}
    for u, v, _ in mst:
        graph[u].append(v)
        graph[v].append(u)

    def dfs(current, target, visited, path):
        if current == target:
            return path
        visited.add(current)
        for nb in graph[current]:
            if nb not in visited:
                result = dfs(nb, target, visited, path + [nb])
                if result:
                    return result
        return None

    route = dfs(start, end, set(), [])
    return route[:-1] if route else []


def sample_edge(A, B, interval):
    """Interpolate points along edge A→B at ~interval metre spacing."""
    dx, dy = B[0] - A[0], B[1] - A[1]
    dist   = (dx**2 + dy**2)**0.5
    steps  = max(1, int(dist // interval))
    return [(A[0] + i * dx / steps, A[1] + i * dy / steps)
            for i in range(steps + 1)]


def xy_to_latlon(x, y, depot_lat, depot_lon, depot_x, depot_y):
    """
    Convert scaled (x,y) metres to lat/lon using depot as geographic origin.
    Subtracts depot scaled coords before projecting so depot maps to
    (depot_lat, depot_lon) exactly.
    """
    dx         = x - depot_x
    dy         = y - depot_y
    lat_offset = dy / 111320.0
    lon_offset = dx / (111320.0 * np.cos(np.radians(depot_lat)))
    return depot_lat + lat_offset, depot_lon + lon_offset


def compute_horizontal_distances(samples):
    return [
        np.hypot(samples[i+1][0] - samples[i][0],
                 samples[i+1][1] - samples[i][1])
        for i in range(len(samples) - 1)
    ]


def compute_slopes(elevations, distances):
    rise = np.diff(np.array(elevations))
    run  = np.array(distances)
    with np.errstate(divide='ignore', invalid='ignore'):
        slopes = np.where(run > 0, rise / run, 0.0)
    return slopes.tolist()


# ============================================================
# ELEVATION
# ============================================================

def extract_elevations(dem_path, latlon_samples):
    """
    Extract elevations from DEM at given (lat, lon) points.
    Uses src.index() + src.read() rather than src.sample()
    which was returning zeros for out-of-bounds points silently.
    """
    with rasterio.open(dem_path) as src:
        data   = src.read(1)
        nodata = src.nodata
        result = []
        for lat, lon in latlon_samples:
            try:
                row, col = src.index(lon, lat)
                # Guard against out-of-bounds indices
                if 0 <= row < data.shape[0] and 0 <= col < data.shape[1]:
                    val = data[row, col]
                    if nodata is not None and val == nodata:
                        result.append(np.nan)
                    else:
                        result.append(float(val))
                else:
                    result.append(np.nan)
            except Exception:
                result.append(np.nan)
        return result


def diagnose_dem(city_name, depot_lat, depot_lon, edge_samples, dem_path):
    """Confirm DEM covers the instance and returns sensible values."""
    with rasterio.open(dem_path) as src:
        print(f"\n  DEM diagnosis: {city_name}")
        print(f"  CRS:    {src.crs}")
        print(f"  Bounds: {src.bounds}")
        print(f"  NoData: {src.nodata}")
        print(f"  Res:    {src.res}")

        # Depot
        in_bounds = (src.bounds.left   <= depot_lon <= src.bounds.right and
                     src.bounds.bottom <= depot_lat <= src.bounds.top)
        print(f"  Depot ({depot_lat:.4f}, {depot_lon:.4f}) in bounds: {in_bounds}")
        if in_bounds:
            row, col = src.index(depot_lon, depot_lat)
            val = src.read(1)[row, col]
            print(f"  Depot elevation: {val:.1f}m")

        # First edge first 3 samples
        print(f"  First edge samples:")
        edge = edge_samples[0]
        elevs = extract_elevations(dem_path, edge["latlon_samples"][:3])
        for (lat, lon), elev in zip(edge["latlon_samples"][:3], elevs):
            print(f"    ({lat:.5f}, {lon:.5f}) → {elev}m")


# ============================================================
# OUTPUT WRITERS
# ============================================================

def write_scaled_instance(df, city_name, scale, benchmark, output_dir):
    output_path = os.path.join(output_dir, f"{city_name}_{scale}_{benchmark}.csv")
    df.to_csv(output_path, index=False)
    print(f"  Written: {output_path}")


def write_short_paths(short_paths, output_path):
    """
    Format: 'from to: step1 step2 ... stepN'
    Matches C++ MST::readSpanningTree expected format.
    Direct MST neighbours have empty step list.
    """
    with open(output_path, 'w') as f:
        for from_site, destinations in short_paths.items():
            for to_site, steps in destinations.items():
                if from_site == to_site:
                    continue
                steps_str = ' '.join(str(s) for s in steps)
                f.write(f"{from_site} {to_site}: {steps_str}\n")
    print(f"  Written: {output_path}")


def write_edge_samples(edge_samples, output_path):
    def clean(obj):
        if isinstance(obj, (np.floating, np.float32, np.float64)):
            return float(obj)
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, (list, tuple)):
            return [clean(i) for i in obj]
        return obj

    serialisable = [{k: clean(v) for k, v in edge.items()}
                    for edge in edge_samples]
    with open(output_path, 'w') as f:
        json.dump(serialisable, f, indent=2)
    print(f"  Written: {output_path}")


# ============================================================
# VISUALISATION
# ============================================================

def visualise_instance(city_name, benchmark, scale,
                       coordinates, mst, edge_samples, png_path):
    all_samples    = []
    all_elevations = []
    for edge in edge_samples:
        all_samples.extend(edge["samples"])
        all_elevations.extend(
            edge.get("elevations", [np.nan] * len(edge["samples"])))

    all_elevations = np.array(all_elevations, dtype=float)
    valid = all_elevations[~np.isnan(all_elevations)]
    if len(valid) == 0:
        print("  No valid elevations to plot")
        return

    elev_min, elev_max = valid.min(), valid.max()
    norm = mcolors.Normalize(vmin=elev_min, vmax=elev_max)
    cmap = cm.terrain

    fig, ax = plt.subplots(figsize=(12, 12))

    for u, v, _ in mst:
        ax.plot([coordinates[u][0], coordinates[v][0]],
                [coordinates[u][1], coordinates[v][1]],
                color='blue', linewidth=1.5, alpha=0.4)

    ax.scatter(coordinates[:, 0], coordinates[:, 1],
               color='black', s=60, zorder=5, label='Sites')

    xs = [s[0] for s in all_samples]
    ys = [s[1] for s in all_samples]
    sc = ax.scatter(xs, ys, c=all_elevations, cmap=cmap, norm=norm,
                    s=15, alpha=0.9, zorder=4, label='Substeps (elevation)')

    cbar = fig.colorbar(sc, ax=ax, fraction=0.03, pad=0.02)
    cbar.set_label("Elevation (m)", fontsize=12)

    ax.set_title(f"{city_name} | {benchmark} | scale={scale} | "
                 f"elev {elev_min:.0f}–{elev_max:.0f}m", fontsize=13)
    ax.set_xlabel("X (scaled, m)")
    ax.set_ylabel("Y (scaled, m)")
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()

    # Save to directory
    plt.savefig(png_path, dpi=300, bbox_inches='tight', format='png')
    print(f"  Saved: {png_path}")

    #plt.show()
    #print(f"  Elevation range: {elev_min:.1f}m – {elev_max:.1f}m")
    plt.close(fig)  # free memory — important for 20x2x15 runs


# ============================================================
# MAIN PIPELINE
# ============================================================

def generate_instance(
    city_name:     str,
    depot_lat:     float,
    depot_lon:     float,
    scale:         int,
    benchmark:     str,
    visualise:     bool = True,
    run_diagnosis: bool = False,
):
    print(f"\n{'='*60}")
    print(f"  {city_name} | {benchmark} | scale={scale}")
    print(f"{'='*60}")

    # --- paths ---
    instance_path     = os.path.join(INSTANCES_DIR, f"{benchmark}.csv")
    city_output_dir   = os.path.join(SCALED_DIR, benchmark, city_name)
    scaled_csv        = os.path.join(city_output_dir, f"{city_name}_{scale}_{benchmark}.csv")
    short_paths_file  = os.path.join(city_output_dir, f"{city_name}_{benchmark}_short_paths_{scale}.txt")
    edge_samples_file = os.path.join(city_output_dir, f"{city_name}_{benchmark}_edge_samples_{scale}.json")
    png_path          = os.path.join(city_output_dir, f"{city_name}_{benchmark}_mst_{scale}.png")

    dem_path          = os.path.join(DEM_DIR, city_name, f"{city_name}_dem.tif")


    os.makedirs(city_output_dir, exist_ok=True)

    # --- 1. Load and normalise raw Solomon instance ---
    if not os.path.exists(instance_path):
        print(f"  SKIPPING: instance not found: {instance_path}")
        return

    df = pd.read_csv(instance_path)
    df.columns = df.columns.str.strip().str.lower()

    # Keep standard Solomon columns only
    df = df[['cust_no.', 'xcoord.', 'ycoord.', 'demand',
             'ready_time', 'due_date', 'service_time']].copy()
    df.columns = ['id', 'x', 'y', 'demand',
                  'ready_time', 'due_time', 'service_time']

    df["scaled_x"] = df["x"] * SCALE_MULTIPLIER
    df["scaled_y"] = df["y"] * SCALE_MULTIPLIER
    coordinates    = df[['scaled_x', 'scaled_y']].values
    n              = len(df)

    # Depot is always site 0
    depot_x = coordinates[0][0]
    depot_y = coordinates[0][1]
    print(f"  Loaded {n} sites | depot scaled: ({depot_x:.1f}, {depot_y:.1f})")

    # --- 2. MST ---
    mst = prim_mst(coordinates)
    print(f"  MST: {len(mst)} edges")

    # --- 3. All-pairs short paths ---
    print(f"  Short paths ({n}x{n})...")
    short_paths = {i: {} for i in range(n)}
    for i in range(n):
        for j in range(n):
            if i != j:
                short_paths[i][j] = compute_route(mst, n, i, j)

    # --- 4. Per-site lat/lon + substep sampling ---
    site_latlons = [
        xy_to_latlon(x, y, depot_lat, depot_lon, depot_x, depot_y)
        for x, y in coordinates
    ]
    df["lat"] = [ll[0] for ll in site_latlons]
    df["lon"] = [ll[1] for ll in site_latlons]

    edge_samples = []
    for u, v, _ in mst:
        A, B    = coordinates[u], coordinates[v]
        samples = sample_edge(A, B, SUBSTEP_INTERVAL)
        latlon_samples = [
            xy_to_latlon(x, y, depot_lat, depot_lon, depot_x, depot_y)
            for x, y in samples
        ]
        edge_samples.append({
            "start":          int(u),
            "end":            int(v),
            "samples":        [(float(x), float(y)) for x, y in samples],
            "latlon_samples": [(float(la), float(lo)) for la, lo in latlon_samples],
        })
    print(f"  Substep sampling: {len(edge_samples)} edges")

    # --- 5. Elevation ---
    if not os.path.exists(dem_path):
        print(f"  WARNING: DEM not found at {dem_path} — elevations skipped")
        df["elevation"] = np.nan
    else:
        if run_diagnosis:
            diagnose_dem(city_name, depot_lat, depot_lon, edge_samples, dem_path)

        # Per-site elevation
        site_elevations = extract_elevations(dem_path, site_latlons)
        df["elevation"] = site_elevations
        print(f"  Site elevation range: "
              f"{np.nanmin(site_elevations):.1f}m – "
              f"{np.nanmax(site_elevations):.1f}m")

        # Per-substep elevation, distances, slopes
        for edge in edge_samples:
            elevations         = extract_elevations(dem_path,
                                                    edge["latlon_samples"])
            edge["elevations"] = elevations
            edge["distances"]  = compute_horizontal_distances(edge["samples"])
            edge["slopes"]     = compute_slopes(elevations, edge["distances"])
        print(f"  Substep elevations extracted")

    # --- 6. Write outputs ---
    write_scaled_instance(df, city_name, scale, benchmark, city_output_dir)
    write_short_paths(short_paths, short_paths_file)
    write_edge_samples(edge_samples, edge_samples_file)

    # --- 7. Visualise ---
    visualise_instance(city_name, benchmark, scale,
                           coordinates, mst, edge_samples, png_path)

    print(f"  Done.")


# ============================================================
# RUN
# ============================================================

# Step 1: ensure all DEMs present and covering the right area
print("="*60)
print("STEP 1: DEM acquisition")
print("="*60)
for i in range(len(cities)):
    city_name = cities.iloc[i]["city"]
    lat       = cities.iloc[i]["latitude"]
    lon       = cities.iloc[i]["longitude"]
    ensure_dem(city_name, lat, lon)

# Step 2: generate all instances
# First instance gets diagnosis + visualisation as a sanity check
print("\n" + "="*60)
print("STEP 2: Instance generation")
print("="*60)
first = True
for i in range(len(cities)):
    city_name = cities.iloc[i]["city"]
    depot_lat = cities.iloc[i]["latitude"]
    depot_lon = cities.iloc[i]["longitude"]

    for scale in scales:
        for benchmark in benchmarks:
            generate_instance(
                city_name     = city_name,
                depot_lat     = depot_lat,
                depot_lon     = depot_lon,
                scale         = scale,
                benchmark     = benchmark,
                visualise     = first,
                run_diagnosis = first,
            )
            first = False